# Real opamps

```{admonition} How To Work
:class: important
The **Preparatory Homework (BAS)** part of this manual should be completed **individually**.

The **Practicum (IC)** part should be completed **in pairs**. Bias and offset numbers are small (millivolts and nanoamps); take turns reading the meter so you spot transcription mistakes early.

Throughout this manual, keep your derivations, datasheet snippets, scope screenshots, and short reflections in your notepad. The comparison tables in the Compare and Conclude section should be filled in your notepad and included in the onepager report for this lab.
```


## Preparatory Homework

### Background
```{admonition} Preparation
:class: tip
You should have completed [Manual 4.1](4.1_opamp_basics.ipynb) before starting this one. We will reuse the non-inverting and inverting circuits you already know how to build, and add three more measurements that probe the **non-ideal** parts of a real opamp.

Re-read the textbook chapter on the non-ideal opamp before you start.

* [Textbook chapter 8: real opamps](../../../theory_part/8_opamp2.md)

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: replace with the published textbook URL or anchor for chapter 8</span>
```

In Manual 4.1 the **golden rules** were enough:

1. The two opamp inputs sit at the same voltage (the virtual short).
2. No current enters the opamp inputs.
3. The output behaves like an ideal voltage source with zero output impedance.
4. The gain is independent of frequency.

A real opamp obeys all four rules **only approximately**. Today you will measure the four imperfections and apply them to a circuit (the integrator) where every single one of them shows up.

```{figure} images/non_ideal_opamp_model.svg
---
name: fig-non-ideal-opamp-model
height: 320px
---
Non-ideal opamp model: an ideal opamp with two bias-current sources (one per input) and an offset voltage source in series with one input.
```

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: draw and add images/non_ideal_opamp_model.svg (ideal triangle, +/-Ibias arrows on the inputs, Voffset source in series)</span>

#### The four non-idealities

* **Input bias current** ($I_{bias}$). Each input draws a small DC current to bias its first transistor stage. For a BJT-input opamp like the LM741, $I_{bias}$ is tens to hundreds of nA. For a FET-input opamp like the TL072, it is around $10\,\mathrm{p A}$. The current must have a path to ground at every input, otherwise the opamp will drift to one of its rails.
* **Input offset voltage** ($V_{offset}$). The two input transistors are not perfectly matched. Even with both inputs tied to ground, the opamp behaves as if there were a small voltage source (a few millivolts) in series with one input. In a high-gain amplifier this offset is amplified by the same factor as the signal.
* **Gain-bandwidth product** (GBW). The open-loop gain is huge at DC (about $10^5$) but it falls off at $-20\,\mathrm{dB/decade}$ above a low corner. The product $\text{gain} \times \text{bandwidth}$ is approximately constant. For the LM741, GBW $\approx 1\,\mathrm{MHz}$; for the TL072, GBW $\approx 3\,\mathrm{MHz}$.

  For the **non-inverting** amplifier, the closed-loop bandwidth is

  $$
  \mathrm{BW} = \frac{\mathrm{GBW}}{1 + R_a/R_b} = \frac{\mathrm{GBW}}{G}.
  $$

  The "+1" is small once the gain is large but matters for low-gain amplifiers.

* **Slew rate** (SR). The output cannot change faster than $\mathrm{SR}$ V/microsecond, regardless of the gain or the bandwidth. For the LM741, SR $\approx 0.5\,\mathrm{V/\mu s}$; for the TL072, SR $\approx 13\,\mathrm{V/\mu s}$. A square wave whose ideal output exceeds this slope comes out as a trapezoid; a sine whose peak slope $2\pi f \hat U$ exceeds the slew rate comes out as a triangle.

```{figure} images/slew_rate_square.svg
---
name: fig-slew-rate-square
height: 220px
---
Square in, trapezoid out: when the demanded slope exceeds the slew rate, the output ramps at a fixed slope.
```

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: draw and add images/slew_rate_square.svg (ideal square + trapezoid + triangle as freq increases)</span>

#### A quick refresher on dB and decade

You will be reading off bandwidths in this manual, which means handling logarithmic scales.

* $1\,\mathrm{decade}$ is a factor of 10 in frequency.
* $1\,\mathrm{octave}$ is a factor of 2.
* A gain expressed in $\mathrm{dB}$ is $20 \log_{10}(|G|)$. So $G = 10$ is $20\,\mathrm{dB}$ and $G = 100$ is $40\,\mathrm{dB}$.
* The **$-3\,\mathrm{dB}$ point** is the frequency at which the closed-loop gain has dropped to $|G|/\sqrt{2} \approx 0.71\,|G|$. Above the $-3\,\mathrm{dB}$ point the amplifier no longer follows the input cleanly.


### Anticipate

The five tasks below let you predict every measurement before you take it. The big skill of this manual is not the algebra, it is the reasoning about which non-ideality dominates in a given circuit. That is exactly what design engineers do every day.


(Task_A1_4_2)=
#### Task A1: Choose $R_a$ to isolate $V_{offset}$
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

You will use a non-inverting amplifier with a single feedback resistor $R_f$ tied to the inverting input and a resistor $R_a$ from the inverting input to ground. The non-inverting input is grounded.

```{figure} images/bias_offset_test_circuit.svg
---
name: fig-bias-offset-test-circuit
height: 320px
---
Test circuit for bias and offset measurement: $R_f = 100\,\mathrm{k}\Omega$ is fixed; $R_a$ is the parameter you choose.
```

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: draw and add images/bias_offset_test_circuit.svg (LM741, +/-12 V, R_f = 100k in feedback, R_a from inverting to GND, non-inverting input to GND through a small R)</span>

The output voltage of this circuit is approximately

$$
U_{out} \approx \left(1 + \frac{R_f}{R_a}\right) V_{offset} + I_{bias} R_f
$$

(where $I_{bias}$ is the bias current of the inverting input).

1. With $R_f = 100\,\mathrm{k}\Omega$ fixed, choose an $R_a$ that makes the **first term dominant**: that is, the offset-voltage term is much larger than the bias-current term. Justify your choice with a one-line inequality, using LM741 typical specs ($V_{offset} \approx 1\,\mathrm{m V}$, $I_{bias} \approx 80\,\mathrm{n A}$).

2. With your chosen $R_a$, compute the predicted $U_{out}$. Is it large enough to be read cleanly with a $3\,\mathrm{1/2}$-digit DMM (resolution $\sim 1\,\mathrm{m V}$ on the 2 V range)?

```{admonition} Check your answer
:class: answer, dropdown
For a small $R_a$ (say $1\,\mathrm{k}\Omega$), the gain factor on $V_{offset}$ is $1 + 100/1 = 101$, so the offset term contributes $\sim 100\,\mathrm{m V}$, while the bias term is $80\,\mathrm{n A} \cdot 100\,\mathrm{k}\Omega = 8\,\mathrm{m V}$. Offset dominates by $\sim 12\times$. The DMM reads $\sim 100\,\mathrm{m V}$ comfortably.
```

(Task_A2_4_2)=
#### Task A2: Choose $R_a$ to isolate $I_{bias}$
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

With the same test circuit, choose an $R_a$ that makes the **second term dominant**: the bias-current term is much larger than the offset term. Justify with the same kind of inequality.

For your chosen $R_a$, predict $U_{out}$ once again.

```{admonition} Check your answer
:class: answer, dropdown
For a large $R_a$ (say $10\,\mathrm{M}\Omega$, so the offset gain is $1 + 100\,\mathrm{k}/10\,\mathrm{M} \approx 1.01$), the offset term contributes only $\sim 1\,\mathrm{m V}$, while the bias term is still $\sim 8\,\mathrm{m V}$. Bias dominates by $\sim 8\times$. The DMM reads $\sim 8\,\mathrm{m V}$, marginal but doable on the 200 mV range.
```

This is exactly the engineering skill you want to take away from this manual: pick the test condition that **isolates** the parameter you want to measure.

(Task_A3_4_2)=
#### Task A3: Predict bandwidth from GBW
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

You will build three non-inverting amplifiers with the LM741: $G \approx 2$ ($R_a = R_b = 1\,\mathrm{k}\Omega$), $G \approx 11$ ($R_a = 10\,\mathrm{k}\Omega$, $R_b = 1\,\mathrm{k}\Omega$), $G \approx 101$ ($R_a = 100\,\mathrm{k}\Omega$, $R_b = 1\,\mathrm{k}\Omega$).

Using the GBW formula from the Background, predict the $-3\,\mathrm{dB}$ frequency for each gain. Fill in the prediction column of the table below; the simulation and measurement columns are for [Task S2](Task_S2_4_2) and [Task I4](Task_I4_4_2).

| $G$ (set) | predicted $\mathrm{BW}$ | simulated $\mathrm{BW}$ | measured $\mathrm{BW}$ | $G \cdot \mathrm{BW}$ |
|---|---|---|---|---|
| 2 |  |  |  |  |
| 11 |  |  |  |  |
| 101 |  |  |  |  |

**Important:** the textbook writes the formula as $\mathrm{BW} = \mathrm{GBW}/G$, but the strict formula for the non-inverting amplifier is $\mathrm{BW} = \mathrm{GBW}/(1 + R_a/R_b)$. The "$1+$" matters for the $G = 2$ row. See [textbook chapter 8](../../../theory_part/8_opamp2.md).

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: replace with the published anchor for the GBW section in chapter 8</span>

(Task_A4_4_2)=
#### Task A4: Predict the slew-limited frequency
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

For an output sine wave $U_{out}(t) = \hat U \sin(2\pi f t)$, the maximum slope is $2\pi f \hat U$. The opamp can keep up with this slope only if $2\pi f \hat U \le \mathrm{SR}$.

1. For $\hat U = 5\,\mathrm{V}$ and the LM741's $\mathrm{SR} = 0.5\,\mathrm{V/\mu s}$, what is the largest $f$ at which the output is **not** slew-limited?
2. Sketch what you expect the output to look like at $f = 10$ times that value: still a sine, a triangle, a trapezoid?
3. Compare with the bandwidth of the $G = 11$ amplifier from Task A3. Which limit kicks in first as you raise the frequency: the bandwidth or the slew rate?

```{admonition} Check your answer
:class: answer, dropdown
$f_{\max} = \mathrm{SR} / (2\pi \hat U) = 0.5 \cdot 10^6 / (2\pi \cdot 5) \approx 16\,\mathrm{kHz}$. At ten times that, the output cannot keep up: the sine becomes a triangle. The $G = 11$ bandwidth is about $1\,\mathrm{MHz}/11 \approx 90\,\mathrm{kHz}$, well above the slew limit at this amplitude. So the slew rate is the active limit at $\hat U = 5\,\mathrm{V}$.
```


(Task_A5_4_2)=
#### Task A5: Predict the integrator slope
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

```{figure} images/integrator.svg
---
name: fig-integrator
height: 280px
---
Inverting integrator: input through $R$, capacitor $C$ in feedback.
```

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: draw and add images/integrator.svg (LM741, R from input to inverting, C in feedback, non-inverting input to GND)</span>

For the integrator in {numref}`fig-integrator`, the output is

$$
U_{out}(t) = -\frac{1}{RC} \int_0^t U_{in}(\tau)\, d\tau + U_{out}(0).
$$

Take $R = 10\,\mathrm{k}\Omega$, $C = 100\,\mathrm{n F}$, and $U_{in}(t)$ a $50\,\mathrm{Hz}$ square wave that switches between $-2\,\mathrm{V}$ and $+2\,\mathrm{V}$.

1. Compute the time constant $\tau = RC$ in your notepad. Compare it with the half-period of the input square wave.
2. While $U_{in} = +2\,\mathrm{V}$, what is $\mathrm{d}U_{out}/\mathrm{d}t$ in V/s? While $U_{in} = -2\,\mathrm{V}$?
3. Sketch one full period of $U_{in}(t)$ and $U_{out}(t)$ on the same time axis. The output should be a triangle. Estimate its peak-to-peak amplitude.

For the integrator derivation, see [textbook chapter 7, section on the integrator](../../../theory_part/7_opamp1.md).

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">🚩 TODO: replace with the published anchor for the integrator section in chapter 7</span>


### Simulate

(Task_S1_4_2)=
#### Task S1: Simulate the bias / offset test circuit

(Task_S2_4_2)=
#### Task S2: Open-loop and closed-loop frequency response

(Task_S3_4_2)=
#### Task S3: The integrator drifts unless you tame it

## Practicum

### Implement and investigate

(Task_I1_4_2)=
#### Task I1: Measure $V_{offset}$

(Task_I2_4_2)=
#### Task I2: Measure $I_{bias}$

(Task_I3_4_2)=
#### Task I3: Class-wide spread

(Task_I4_4_2)=
#### Task I4: GBW from a gain sweep

(Task_I5_4_2)=
#### Task I5: Slew rate from a square wave

(Task_I6_4_2)=
#### Task I6: Build the integrator

(Task_I7_4_2)=
#### Task I7: Vary $R$ or $C$ by a decade

(Task_I8_4_2)=
#### Task I8: Tame the integrator

### Compare and conclude

(Task_C1_4_2)=
#### Task C1: Bias and offset, three columns

(Task_C2_4_2)=
#### Task C2: Gain-bandwidth product as a constant

(Task_C3_4_2)=
#### Task C3: Re-state the golden rules as inequalities

(Task_C4_4_2)=
#### Task C4: A design decision